# Test propre de YOLOv1 sur une image

Ce notebook propose une version plus lisible et reproductible du test d'inférence **YOLOv1 / Darknet** sur une image unique.

## Objectif
- vérifier que l'environnement Darknet fonctionne ;
- charger la configuration et les poids YOLOv1 ;
- lancer une prédiction sur une image ;
- afficher le fichier de sortie généré par Darknet.

## Ce que ce notebook améliore
- paramètres centralisés ;
- vérifications de fichiers claires ;
- fonctions utilitaires réutilisables ;
- étapes optionnelles bien séparées (patch, compilation, téléchargement des poids) ;
- exécution plus sûre et plus facile à relire.


## 1) Imports et configuration

Modifie uniquement les variables de la cellule suivante si nécessaire.


In [ ]:
from pathlib import Path
from IPython.display import Image, display, Markdown
import subprocess
import shlex
import os


In [ ]:
PROJECT_ROOT = Path(".").resolve()

DARKNET_BIN = PROJECT_ROOT / "darknet"
CFG_PATH = PROJECT_ROOT / "cfg" / "yolov1.cfg"
WEIGHTS_PATH = PROJECT_ROOT / "yolov1.weights"

IMAGE_PATH = PROJECT_ROOT / "data" / "dog_bicycle_car.jpg"

THRESH = 0.25
USE_PATCH_TO_SHOW_CONFIDENCE = True
RECOMPILE_DARKNET = False
AUTO_DOWNLOAD_WEIGHTS = False
SHOW_STDOUT = True

print("PROJECT_ROOT :", PROJECT_ROOT)
print("CFG_PATH     :", CFG_PATH)
print("WEIGHTS_PATH :", WEIGHTS_PATH)
print("IMAGE_PATH   :", IMAGE_PATH)


## 2) Fonctions utilitaires


In [ ]:
import re

def run_command(command: str, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    """Exécute une commande shell et affiche proprement sa sortie."""
    print(f"$ {command}")
    result = subprocess.run(
        shlex.split(command),
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True
    )
    if SHOW_STDOUT and result.stdout.strip():
        print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Commande échouée avec code {result.returncode}: {command}")
    return result


def assert_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} introuvable : {path}")


def find_prediction_file(project_root: Path) -> Path:
    candidates = [
        project_root / "predictions.png",
        project_root / "predictions.jpg",
        project_root / "predictions.jpeg",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Aucun fichier predictions.png / predictions.jpg / predictions.jpeg trouvé."
    )


def patch_image_c_to_show_confidence(project_root: Path) -> None:
    """Patch optionnel de src/image.c pour afficher 'classe + probabilité'."""
    image_c = project_root / "src" / "image.c"
    assert_exists(image_c, "Fichier source image.c")

    marker = "/* PATCH_SHOW_CONFIDENCE_IN_LABELSTR */"
    text = image_c.read_text(encoding="utf-8", errors="ignore")

    if marker in text:
        print("Patch déjà présent.")
        return

    pattern = r'strcat\(labelstr,\s*names\[j\]\s*\);'
    replacement = (
        r'do { char tmp[256]; '
        r'sprintf(tmp, "%s %.0f%%", names[j], dets[i].prob[j]*100); '
        r'strcat(labelstr, tmp); } while(0); ' + marker
    )

    matches = len(re.findall(pattern, text))
    if matches == 0:
        raise RuntimeError(
            "Pattern non trouvé dans src/image.c. Vérifie la version de Darknet utilisée."
        )

    new_text = re.sub(pattern, replacement, text)
    image_c.write_text(new_text, encoding="utf-8")
    print(f"Patch appliqué ({matches} remplacement(s))")


def maybe_download_weights(weights_path: Path) -> None:
    if weights_path.exists() and weights_path.stat().st_size > 0:
        print("Poids déjà présents.")
        return

    if not AUTO_DOWNLOAD_WEIGHTS:
        raise FileNotFoundError(
            f"Poids introuvables : {weights_path}\n"
            "Place yolov1.weights à la racine du projet ou active AUTO_DOWNLOAD_WEIGHTS = True."
        )

    urls = [
        "https://pjreddie.com/media/files/yolov1.weights",
        "http://pjreddie.com/media/files/yolov1.weights",
    ]

    for url in urls:
        print(f"Tentative de téléchargement : {url}")
        result = subprocess.run(
            ["wget", "-O", str(weights_path), url],
            capture_output=True,
            text=True
        )
        if result.returncode == 0 and weights_path.exists() and weights_path.stat().st_size > 0:
            print("Téléchargement terminé.")
            return

    raise RuntimeError("Impossible de télécharger yolov1.weights automatiquement.")


def compile_darknet(project_root: Path) -> None:
    run_command("make clean", cwd=project_root, check=False)
    run_command("make all", cwd=project_root, check=True)


def run_yolov1_test(
    darknet_bin: Path,
    cfg_path: Path,
    weights_path: Path,
    image_path: Path,
    thresh: float = 0.25,
) -> subprocess.CompletedProcess:
    assert_exists(darknet_bin, "Binaire darknet")
    assert_exists(cfg_path, "Fichier cfg")
    assert_exists(weights_path, "Fichier weights")
    assert_exists(image_path, "Image de test")

    command = (
        f"{darknet_bin} yolo test "
        f"{cfg_path} {weights_path} {image_path} -thresh {thresh}"
    )
    return run_command(command, cwd=darknet_bin.parent)


## 3) Préparation de l'environnement


In [ ]:
if USE_PATCH_TO_SHOW_CONFIDENCE:
    patch_image_c_to_show_confidence(PROJECT_ROOT)

if RECOMPILE_DARKNET:
    compile_darknet(PROJECT_ROOT)

maybe_download_weights(WEIGHTS_PATH)

assert_exists(CFG_PATH, "Configuration YOLOv1")
assert_exists(WEIGHTS_PATH, "Poids YOLOv1")
assert_exists(IMAGE_PATH, "Image de test")

print("Environnement prêt ✅")


## 4) Lancer l'inférence sur une image


In [ ]:
result = run_yolov1_test(
    darknet_bin=DARKNET_BIN,
    cfg_path=CFG_PATH,
    weights_path=WEIGHTS_PATH,
    image_path=IMAGE_PATH,
    thresh=THRESH,
)

print("Inférence terminée ✅")


## 5) Afficher le résultat


In [ ]:
prediction_path = find_prediction_file(PROJECT_ROOT)

display(Markdown(f"**Fichier généré :** `{prediction_path}`"))
display(Image(filename=str(prediction_path)))
